In [1]:
!uv pip install biopython transformers==5.17.0

Using Python 3.13.15 environment at: /usr
Resolved 28 packages in 632ms
Prepared 2 packages in 756ms
Uninstalled 1 package in 1.10s
Installed 2 packages in 119ms
 + biopython==1.88
 - transformers==5.16.1
 + transformers==5.17.0


In [2]:
!wget https://raw.githubusercontent.com/ArcInstitute/evo2/main/scripts/gene_completion/data/prokaryote_genes.csv

--2026-09-15 17:04:06--  https://raw.githubusercontent.com/ArcInstitute/evo2/main/scripts/gene_completion/data/prokaryote_genes.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 29828 (29K) [text/plain]
Saving to: ‘prokaryote_genes.csv’

prokaryote_genes.cs 100%[===================>]  29.13K  --.-KB/s    in 0.001s  

2026-09-15 17:04:06 (19.0 MB/s) - ‘prokaryote_genes.csv’ saved [29828/29828]



In [1]:
import csv
import os
import subprocess
import threading
import time
import torch
from Bio.Align import PairwiseAligner, substitution_matrices
from Bio.Seq import Seq
from transformers import AutoModelForCausalLM, AutoTokenizer

HF_DIR = "Aquiles-ai/Evo2-1B-Base"
DATA_CSV = "./prokaryote_genes.csv"
OUT_DIR = "./out_gene_completion"
TAG = "evo2_1b_base"
GENES = ""  # comma-separated subset, or "" for all
N_GEN = 5
TEMPERATURE = 0.7
TOP_K = 4
SEED = 0
PROMPT_FRACTION = 0.30
PROK_UPSTREAM_LEN = 1000
MONITOR_EVERY_S = 15  # background GPU/progress line cadence, 0 = off

os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE.startswith("cuda") else torch.float32

PROG = {"gene": "", "idx": -1, "tokens": 0, "t_start": 0.0, "t_last": 0.0}

class _CountStreamer:
    def put(self, value):
        n = value.numel() if isinstance(value, torch.Tensor) else len(value)
        PROG["tokens"] += int(n)
        PROG["t_last"] = time.time()

    def end(self):
        PROG["t_last"] = time.time()

_STREAMER = _CountStreamer()
_MON_STOP = threading.Event()

def _gpu_line():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10,
        ).stdout.strip().split(",")
        util, used, total, temp = [x.strip() for x in out]
        return f"gpu={util}% vram={used}/{total}MiB {temp}C"
    except Exception:
        return "gpu=n/a"

def _monitor_loop(every_s):
    t0 = time.time()
    last_n = 0
    while not _MON_STOP.wait(every_s):
        now = time.time()
        n = PROG["tokens"]
        if n < last_n:
            last_n = n
        rate = (n - last_n) / every_s
        last_n = n
        idle = now - PROG["t_last"] if PROG["t_last"] else 0.0
        flag = " STALL?" if PROG["t_start"] and idle > 2 * every_s else ""
        own = ""
        if DEVICE.startswith("cuda"):
            own = f" self={torch.cuda.memory_allocated() / 2 ** 20:.0f}MiB"
        print(f"[mon {now - t0:7.0f}s] {PROG['gene']} gen{PROG['idx']} "
              f"tok={n} {rate:4.1f}tok/s idle={idle:4.0f}s | {_gpu_line()}{own}{flag}",
              flush=True)

def start_monitor():
    if not MONITOR_EVERY_S:
        return None
    PROG["t_start"] = time.time()
    PROG["t_last"] = time.time()
    th = threading.Thread(target=_monitor_loop, args=(MONITOR_EVERY_S,), daemon=True)
    th.start()
    return th

def prokaryote_prompt(genomic, cds_start=5000, upstream_len=PROK_UPSTREAM_LEN,
                      fraction=PROMPT_FRACTION):
    genomic = "".join(genomic.split()).upper()
    coding_nt = len(genomic) - cds_start
    coding_aa_take = round(coding_nt / 3.0 * fraction)
    take_nt = coding_aa_take * 3
    start = max(0, cds_start - upstream_len)
    prompt = genomic[start:cds_start] + genomic[cds_start:cds_start + take_nt]
    return prompt, coding_aa_take

def translate_dna(dna, to_stop=True):
    dna = "".join(dna.split()).upper()
    usable = len(dna) - (len(dna) % 3)
    if usable <= 0:
        return ""
    return str(Seq(dna[:usable]).translate(to_stop=to_stop))

_ALIGNER = None

def _aligner():
    global _ALIGNER
    if _ALIGNER is None:
        a = PairwiseAligner()
        a.substitution_matrix = substitution_matrices.load("BLOSUM62")
        a.open_gap_score = -11
        a.extend_gap_score = -1
        a.mode = "global"
        _ALIGNER = a
    return _ALIGNER

def aligned_identity_after(query_aa, ref_aa, ref_start):
    if not query_aa or not ref_aa:
        return float("nan")
    aln = _aligner().align(ref_aa, query_aa)[0]
    matches = aligned = 0
    ref_row, qry_row = aln.indices
    for r, q in zip(ref_row, qry_row):
        if r >= ref_start and q >= 0:
            aligned += 1
            if ref_aa[r] == query_aa[q]:
                matches += 1
    if aligned == 0:
        return float("nan")
    return 100.0 * matches / aligned

def score_prokaryote(generated_full, reference_protein, upstream_len, prompt_cds_aa):
    coding = generated_full[upstream_len:]
    gen_protein = translate_dna(coding, to_stop=True)
    return aligned_identity_after(gen_protein, reference_protein, prompt_cds_aa)

def complete_once(model, tok, prompt, n_tokens, seed):
    torch.manual_seed(seed)
    ids = torch.tensor([tok.vortex_tokenize(prompt)], dtype=torch.long, device=DEVICE)
    PROG["tokens"] = 0
    PROG["t_last"] = time.time()
    with torch.inference_mode():
        gen = model.generate(
            ids, max_new_tokens=n_tokens, do_sample=True,
            temperature=TEMPERATURE, top_k=TOP_K,
            use_cache=True, pad_token_id=tok.pad_token_id,
            streamer=_STREAMER if MONITOR_EVERY_S else None,
        )
    full = "".join(tok.vortex_detokenize(gen[0].tolist()).split()).upper()
    if not full.startswith(prompt):
        full = prompt + full
    return full

def main():
    tok = AutoTokenizer.from_pretrained(HF_DIR, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        HF_DIR, trust_remote_code=True, dtype=DTYPE).to(DEVICE).eval()
    max_pos = model.config.max_position_embeddings

    with open(DATA_CSV, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    if GENES:
        wanted = {g.strip().lower() for g in GENES.split(",")}
        rows = [r for r in rows if r["gene"].strip().lower() in wanted]
    if not rows:
        raise SystemExit("No genes selected.")

    raw_path = os.path.join(OUT_DIR, f"{TAG}_prokaryote_completions.csv")
    gene_means = {}
    start_monitor()
    with open(raw_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["gene", "organism", "gen_idx", "aa_recovery_non_prompt", "prompt_len", "completion"])
        for row in rows:
            gene = row["gene"]
            genomic = "".join(str(row["genomic_sequence"]).split()).upper()
            ref_protein = str(row["reference_protein"]).strip()
            prompt, prompt_cds_aa = prokaryote_prompt(genomic, int(row["cds_start"]))
            n_tokens = max(1, len(ref_protein) - prompt_cds_aa) * 3 + 150
            if len(prompt) + n_tokens > max_pos:
                raise SystemExit(
                    f"[{gene}] prompt+n_tokens exceeds context ({max_pos})")
            print(f"[{gene}] prompt={len(prompt)} nt, n_tokens={n_tokens}, gens={N_GEN}")
            scores = []
            for g in range(N_GEN):
                PROG["gene"] = gene
                PROG["idx"] = g
                full = complete_once(model, tok, prompt, n_tokens, SEED + g)
                rec = score_prokaryote(full, ref_protein, PROK_UPSTREAM_LEN, prompt_cds_aa)
                scores.append(rec)
                w.writerow([gene, row.get("organism", ""), g, f"{rec:.2f}", len(prompt), full])
                print(f"  gen {g}: AA recovery={rec:.2f}%")
            gene_means[gene] = sum(scores) / len(scores)

    stats_path = os.path.join(OUT_DIR, f"{TAG}_prokaryote_per_gene_stats.csv")
    with open(stats_path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["gene", "n_samples", "mean_aa_recovery"])
        for gene, mean in gene_means.items():
            w.writerow([gene, N_GEN, f"{mean:.2f}"])
    panel = sum(gene_means.values()) / len(gene_means)
    _MON_STOP.set()
    print(f"\nWrote {raw_path}\nWrote {stats_path}")
    print(f"Panel mean AA recovery: {panel:.2f}% (ref 1B base: 64.9)")

if __name__ == "__main__":
    main()


Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

[ftsZ] prompt=1342 nt, n_tokens=948, gens=5
[mon      15s] ftsZ gen0 tok=1548 103.2tok/s idle=   0s | gpu=23% vram=2973/15360MiB 53C self=2202MiB
[mon      30s] ftsZ gen0 tok=1962 27.6tok/s idle=   0s | gpu=47% vram=2973/15360MiB 60C self=2213MiB
  gen 0: AA recovery=37.84%
[mon      45s] ftsZ gen1 tok=1519  0.0tok/s idle=   0s | gpu=59% vram=2973/15360MiB 66C self=2202MiB
[mon      60s] ftsZ gen1 tok=2080 37.4tok/s idle=   0s | gpu=59% vram=2973/15360MiB 71C self=2216MiB
  gen 1: AA recovery=38.24%
[mon      75s] ftsZ gen2 tok=1638  0.0tok/s idle=   0s | gpu=58% vram=2973/15360MiB 75C self=2203MiB
[mon      90s] ftsZ gen2 tok=2172 35.6tok/s idle=   0s | gpu=47% vram=2973/15360MiB 79C self=2218MiB
  gen 2: AA recovery=28.81%
[mon     105s] ftsZ gen3 tok=1437  0.0tok/s idle=   0s | gpu=81% vram=2973/15360MiB 77C self=2202MiB
[mon     120s] ftsZ gen3 tok=1986 36.6tok/s idle=   0s | gpu=61% vram=2973/15360MiB 81C self=2213MiB
  gen 3: AA recovery=35.29%
[mon     135s] ftsZ gen4 tok=1524  